# notebook 05 - task 2.3: error analysis and cluster quality reflection

**assignment requirement:** identify the pair of cluster labels your classifier
confuses most often, then answer:
1. which two clusters are most often confused, and in which direction?
2. what is the main cause of this confusion? (quantitative or textual evidence)
3. what does this confusion tell you about the quality of your clustering?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from scipy.stats import mannwhitneyu

import sys
sys.path.append("..")
from src.config.settings import (
    FIG_DIR,
    EFFORT_DIMENSIONS,
    EFFORT_DIM_LABELS,
    CLUSTER_PALETTE,
    AGENT_ORDER,
    AGENT_DISPLAY,
    RANDOM_STATE,
    TEST_SIZE,
    TFIDF_MAX_FEATURES,
    TFIDF_NGRAM_RANGE,
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
)
from src.utils.helper import (
    load_clustered,
    prepare_text_data,
    preprocess_text,
    CLUSTER_LABEL_MAP,
)

FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_style("whitegrid")

## 1. reproduce classification results

In [ ]:
df = load_clustered()
df_text = prepare_text_data(df)
df_text["body_clean"] = preprocess_text(df_text["body"])
df_text.head()

In [ ]:
X = df_text["body_clean"].to_numpy().astype(str)
y = df_text["cluster"].to_numpy().astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, solver="lbfgs")
lr.fit(X_train_tfidf, y_train)
y_pred = lr.predict(X_test_tfidf)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("confusion matrix:")
print(cm)

## 2. identify the most confused pair (question 1)

In [ ]:
n_classes = cm.shape[0]
pair_confusion = []
for i in range(n_classes):
    for j in range(i + 1, n_classes):
        bidir = cm[i, j] + cm[j, i]
        pair_confusion.append({
            "pair": f"{i} <-> {j}",
            "label_pair": f"{CLUSTER_LABEL_MAP[i]} <-> {CLUSTER_LABEL_MAP[j]}",
            "i_to_j": cm[i, j],
            "j_to_i": cm[j, i],
            "bidirectional": bidir,
        })

pair_df = pd.DataFrame(pair_confusion).sort_values("bidirectional", ascending=False)
pair_df

In [ ]:
worst = pair_df.iloc[0]
print(f"most confused pair: cluster {worst['pair']}")

In [ ]:
print(f"{worst['label_pair']}")

In [ ]:
print(f"cluster 1 -> 3: {int(worst['i_to_j'])} misclassified")

In [ ]:
print(f"cluster 3 -> 1: {int(worst['j_to_i'])} misclassified")

In [ ]:
print(f"bidirectional total: {int(worst['bidirectional'])}")

the most confused pair is clusters 1 and 3 (auto-merged small <-> auto-merged large), the dominant direction is cluster 3 -> cluster 1 (369 misclassifications), meaning the classifier often predicts 'auto-merged small' when the true, label is 'auto-merged large'. the reverse (1 -> 3) accounts for 281 errors.

together these 650 errors represent 14.5% of all test predictions

## 3. root cause analysis (question 2)

why does the classifier confuse clusters 1 and 3? both share near-zero review
activity (0 formal reviews, 0 comments, 0 reviewers). the only distinguishing
dimension is `churn_per_review_cycle` (median 16 vs 165 lines). we investigate
three hypotheses:

1. **textual overlap:** pr descriptions for small and large auto-merged prs
   use similar language (fix, feat, docs), making tf-idf features ambiguous.
2. **boundary prs:** misclassified prs sit near the cluster boundary in the
   churn dimension, meaning the clustering itself is unstable at the border.
3. **agent confound:** both clusters are dominated by openai codex, so agent-
   specific language patterns do not help discrimination.

### hypothesis 1: textual overlap - task type distribution

reconstruct test set with original indices

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y))
df_test = df_text.iloc[test_idx].copy()
df_test["y_pred"] = y_pred

c1_correct = df_test[(df_test["cluster"] == 1) & (df_test["y_pred"] == 1)]
print(f"cluster 1 correctly classified: {len(c1_correct):,}")


In [ ]:
c3_correct = df_test[(df_test["cluster"] == 3) & (df_test["y_pred"] == 3)]
print(f"cluster 3 correctly classified: {len(c3_correct):,}")


In [ ]:
c1_as_c3 = df_test[(df_test["cluster"] == 1) & (df_test["y_pred"] == 3)]
print(f"cluster 1 misclassified as 3:  {len(c1_as_c3):,}")


In [ ]:
c3_as_c1 = df_test[(df_test["cluster"] == 3) & (df_test["y_pred"] == 1)]
print(f"cluster 3 misclassified as 1:  {len(c3_as_c1):,}")

task type distribution: correctly classified vs misclassified prs

In [ ]:
task_compare = pd.DataFrame({
    "c1 correct": c1_correct["task_type"].value_counts(normalize=True),
    "c3 correct": c3_correct["task_type"].value_counts(normalize=True),
    "c3->c1 error": c3_as_c1["task_type"].value_counts(normalize=True),
    "c1->c3 error": c1_as_c3["task_type"].value_counts(normalize=True),
}).fillna(0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
task_compare[["c1 correct", "c3 correct", "c3->c1 error", "c1->c3 error"]].plot(
    kind="bar", ax=ax, color=[CLUSTER_PALETTE[1], CLUSTER_PALETTE[3], "#999999", "#666666"],
)
ax.set_title("task type distribution: correctly classified vs misclassified prs")
ax.set_ylabel("proportion")
ax.set_xlabel("task type")
ax.legend()
fig.savefig(FIG_DIR / "05_task_type_misclassification.png", dpi=150, bbox_inches="tight")
plt.show()

task type distributions:

In [ ]:
print(task_compare.round(3).to_string())


key observation: cluster 3 (auto-merged large) is 60% feat, while cluster 1, (auto-merged small) is only 26% feat. however, the misclassified c3->c1 group, has a feat proportion closer to cluster 1 than cluster 3, suggesting that, non-feat prs in cluster 3 are harder to distinguish from cluster 1 prs

### hypothesis 2: boundary prs - churn distribution of misclassified samples

cluster 1 vs cluster 3: churn distribution

churn distributions: correct vs misclassified prs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, dim in zip(axes, ["churn_per_review_cycle", "total_churn"]):
    ax.hist(c1_correct[dim], bins=50, alpha=0.5, color=CLUSTER_PALETTE[1], label="c1 correct", density=True)
    ax.hist(c3_correct[dim], bins=50, alpha=0.5, color=CLUSTER_PALETTE[3], label="c3 correct", density=True)
    ax.hist(c3_as_c1[dim], bins=50, alpha=0.5, color="#999999", label="c3->c1 error", density=True)
    ax.set_title(f"{dim} distribution")
    ax.set_xlabel(dim)
    ax.set_ylabel("density")
    ax.legend()
    ax.set_xlim(0, 500)

fig.suptitle("churn distributions: correct vs misclassified prs", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_churn_misclassification.png", dpi=150, bbox_inches="tight")
plt.show()

churn_per_review_cycle statistics:

In [ ]:
compare_churn = pd.DataFrame({
    "c1 correct": c1_correct["churn_per_review_cycle"].describe(),
    "c3 correct": c3_correct["churn_per_review_cycle"].describe(),
    "c3->c1 error": c3_as_c1["churn_per_review_cycle"].describe(),
    "c1->c3 error": c1_as_c3["churn_per_review_cycle"].describe(),
})
print(compare_churn.round(2).to_string())

quantitative evidence: mann-whitney u test on churn for c3 correct vs c3->c1 error

In [ ]:
stat, p = mannwhitneyu(
    c3_correct["churn_per_review_cycle"],
    c3_as_c1["churn_per_review_cycle"],
    alternative="two-sided",
)

mann-whitney u test: c3 correct vs c3->c1 error

In [ ]:
print(f"u-statistic: {stat:.1f}")
print(f"p-value: {p:.2e}")

interpretation: the misclassified c3->c1 prs have significantly lower churn, than correctly classified c3 prs. they sit near the cluster boundary, having code volume that is ambiguous between 'small' and 'large'. since churn is not captured in the text, the classifier cannot distinguish them.

### hypothesis 3: textual evidence - sample misclassified prs

sample misclassified prs (cluster 3 predicted as cluster 1):

In [ ]:
for i, row in c3_as_c1.head(5).iterrows():
    print(f"churn_per_review_cycle: {row['churn_per_review_cycle']:.0f}")
    print(f"total_churn: {row['total_churn']:.0f}")
    print(f"task_type: {row['task_type']}")
    print(f"body (first 200 chars): {row['body_clean'][:200]}")
    print("-" * 40)

sample misclassified prs (cluster 1 predicted as cluster 3):

In [ ]:
for i, row in c1_as_c3.head(5).iterrows():
    print(f"\nchurn_per_review_cycle: {row['churn_per_review_cycle']:.0f}")
    print(f"total_churn: {row['total_churn']:.0f}")
    print(f"task_type: {row['task_type']}")
    print(f"body (first 200 chars): {row['body_clean'][:200]}")
    print("-" * 40)

## 4. root cause summary (question 2 answer)

**the most confused pair is clusters 1 (auto-merged small) and 3 (auto-merged large),
with 650 bidirectional misclassifications (14.5% of the test set). the dominant
direction is cluster 3 -> cluster 1 (369 errors).**

**main cause: the two clusters differ only in code volume, not in review activity
or textual content.** both have zero formal reviews, zero comments, zero reviewers,
and near-zero duration. the sole distinguishing dimension is `churn_per_review_cycle`
(median 16 vs 165 lines), which is a code-level metric that is not reflected in
pr descriptions.

**quantitative evidence:**
- the mann-whitney u test confirms that misclassified c3->c1 prs have
  significantly lower churn than correctly classified c3 prs (p < 0.001).
  they sit near the cluster boundary in churn space.
- task type overlap: cluster 1 is 26% feat while cluster 3 is 60% feat,
  but the misclassified c3->c1 group has a feat proportion closer to cluster 1,
  making their text harder to distinguish.

**textual evidence:**
- misclassified pr descriptions use similar language (fix, update, add) regardless
  of code volume. a pr that adds 200 lines of a new feature and a pr that fixes
  20 lines of a bug can both start with 'fix' or 'add' in their description.


## 5. cluster quality reflection (question 3)

the high confusion between clusters 1 and 3 reveals a structural weakness in
our clustering solution: **kmeans separated two groups that are only
distinguishable by code volume, not by any textual or semantic signal.**

this has three implications for clustering quality:

1. **the 4-cluster solution may be over-splitting the 'no review' population.**
   clusters 1 and 3 together represent 81% of all prs and are separated
   only by churn threshold. a 3-cluster solution (merging 1 and 3) would
   produce more textually distinguishable groups, though at the cost of
   losing the practically important distinction between small and large
   auto-merged prs.

2. **the clustering captures structurally different types of variation.**
   clusters 0 and 2 are separated by genuine review behavior (iterations,
   depth, breadth) which may be reflected in pr descriptions. clusters 1
   and 3 are separated only by code volume, which is invisible to a text
   classifier. the high confusion validates that churn-based separation
   is structurally different from review-behavior-based separation.

3. **the clustering is still meaningful despite the confusion.** the silhouette
   score (0.395) and dbi (0.997) confirm genuine structure. the issue is not
   that the clusters are arbitrary, but that one distinguishing dimension
   (churn) is not textually recoverable. from a practical standpoint, the
   distinction between 'small auto-merged' and 'large auto-merged' is still
   valuable - it flags that 45% of prs are large code additions with zero
   review, a serious code quality concern.

visualize the churn boundary between clusters 1 and 3

In [ ]:
c1_all = df_text[df_text["cluster"] == 1]
c3_all = df_text[df_text["cluster"] == 3]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log1p(c1_all["churn_per_review_cycle"]), bins=80, alpha=0.5, color=CLUSTER_PALETTE[1], label="cluster 1 (auto-merged small)", density=True)
ax.hist(np.log1p(c3_all["churn_per_review_cycle"]), bins=80, alpha=0.5, color=CLUSTER_PALETTE[3], label="cluster 3 (auto-merged large)", density=True)
ax.set_title("churn_per_review_cycle distribution: clusters 1 vs 3 (log1p scale)")
ax.set_xlabel("log1p(churn_per_review_cycle)")
ax.set_ylabel("density")
ax.legend()
ax.axvline(np.log1p(55), color="red", linestyle="--", alpha=0.7, label="approximate boundary")
ax.legend()
fig.savefig(FIG_DIR / "05_churn_boundary_1v3.png", dpi=150, bbox_inches="tight")
plt.show()